# Milestone 2 — Transformer-only against Diffusion

*Optional extensions: transformer-only vs diffusion-based generation, and the BERT-conditioned arm.*

Three ways to turn the **same ten captions** into pictures:

| Arm | Encoder | How the image is made | Size |
|---|---|---|---|
| **SD 1.5** | CLIP ViT-L/14 | start from noise, denoise ~20 times — continuous latents | 512 |
| **aMUSEd** | CLIP ViT-L/14 | chop into discrete tiles, mask them all, fill in the blanks over ~12 rounds | 256 |
| **LDM-BERT** | BERT | latent diffusion, conditioned on BERT instead of CLIP | 256 |

**SD 1.5 against aMUSEd is the cleanest experiment here**: both use the *same* CLIP ViT-L/14 text
encoder, so the encoder is held fixed and the only variable is the generative mechanism. aMUSEd has
no noise, no noise schedule and no samplers at all — everything the sweep notebook spent its time
tuning simply does not exist.

**LDM-BERT changes the encoder instead**, which is why it reads as a separate axis rather than a
fourth point on the same line.

**What does not transfer:** the fixed-latent protocol. aMUSEd has no starting noise to hold still —
its randomness is in which tokens it commits to first — so the arms share the **prompts and the
seed**, not the latents.

Each arm is freed before the next loads.

---
## 1. Setup

This notebook downloads roughly 7 GB of aMUSEd and LDM-BERT weights, so it needs a session with
internet access, or those weights attached as a second Dataset with `HF_HOME` pointing at it.
Artefacts go to `/kaggle/working/outputs/`.

In [ ]:
import json, inspect, math, re, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

import diffusers
import transformers
from diffusers import (AmusedPipeline, DPMSolverMultistepScheduler, LDMTextToImagePipeline,
                       StableDiffusionPipeline)
from transformers import CLIPImageProcessor, CLIPModel, CLIPTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()


def resolve_paths():
    if Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir():
        found = sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))
        if not found:
            raise RuntimeError(
                "Attach the Dataset holding processed/pairs.parquet. Under /kaggle/input: "
                + (", ".join(p.name for p in Path("/kaggle/input").iterdir()) or "nothing"))
        data, out = Path("/kaggle/working/data"), Path("/kaggle/working/outputs")
        if not (data / "processed").is_dir():
            print(f"copying {found[0].parent} -> {data / 'processed'} ...")
            shutil.copytree(found[0].parent, data / "processed")
        return data, out
    return Path("data"), Path("outputs")


DATA_DIR, OUT_DIR = resolve_paths()
PROCESSED = DATA_DIR / "processed"
SWEEP_DIR, PLOTS_DIR, TABLES_DIR = OUT_DIR / "sweep", OUT_DIR / "plots", OUT_DIR / "tables"
for d in (SWEEP_DIR, PLOTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SD_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

SEED = 42
N_PROMPTS = 10
PROMPT_MIN_WORDS, PROMPT_MAX_WORDS = 8, 18
BATCH_SIZE = 4
GUIDANCE = 7.5
SD_STEPS, AMUSED_STEPS, LDM_STEPS = 20, 12, 50

CLIP_JOINT_MODEL = "openai/clip-vit-base-patch32"
CLIP_MAX_TOKENS = 77
SD_REPOS = ["stable-diffusion-v1-5/stable-diffusion-v1-5",
            "sd-legacy/stable-diffusion-v1-5",
            "CompVis/stable-diffusion-v1-4"]
AMUSED_REPO, LDM_REPO = "amused/amused-256", "CompVis/ldm-text2im-large-256"

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"torch {torch.__version__} | diffusers {diffusers.__version__}")
print(f"device  : {DEVICE}")
print(f"data    : {PROCESSED}")
print(f"outputs : {OUT_DIR}")

In [ ]:
manifest = pd.read_parquet(PROCESSED / "pairs.parquet")
eligible = manifest[manifest["n_words"].between(PROMPT_MIN_WORDS, PROMPT_MAX_WORDS)]
picks = eligible.sample(n=N_PROMPTS, random_state=SEED).sort_values("pair_idx").reset_index(drop=True)

PROMPTS = picks["caption"].tolist()
REAL_IMAGES = [PROCESSED / "images_512" / Path(p).name for p in picks["image_512_path"]]

print(f"{len(PROMPTS)} prompts, seed {SEED}\n")
for i, caption in enumerate(PROMPTS):
    print(f"  {i:>2}  {caption}")

### Scoring, and running an arm

One metric across all three: CLIP(text, generated), in CLIP ViT-B/32's joint space — a *different*
CLIP from the ViT-L/14 that conditions SD and aMUSEd, so the judge is not one of the contestants'
own encoders. Resolution does not bias it, because CLIP resizes everything to 224 before looking,
so the 256px arms are not penalised for being small.

What a bare cosine misses is image quality: a sharp image and a muddy one that both contain the
right objects score alike. Read the grid alongside the table.

`run_arm` passes only the keyword arguments a pipeline actually declares. The three do not share a
call signature, and LDM's ends in `**kwargs`, which would swallow a wrong argument silently rather
than raising. It also batches, because classifier-free guidance doubles the effective batch and ten
512px prompts at once does not fit a small card.

In [ ]:
clip_tokenizer = CLIPTokenizer.from_pretrained(CLIP_JOINT_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_JOINT_MODEL).to(DEVICE, torch.float32).eval()
clip_processor = CLIPImageProcessor.from_pretrained(CLIP_JOINT_MODEL)


def pooled(out):
    return out.pooler_output if hasattr(out, "pooler_output") else out


@torch.no_grad()
def embed_images(images, batch_size=32):
    out = []
    for i in range(0, len(images), batch_size):
        batch = [(img if isinstance(img, Image.Image) else Image.open(img)).convert("RGB")
                 for img in images[i:i + batch_size]]
        px = clip_processor(images=batch, return_tensors="pt").to(DEVICE)
        out.append(pooled(clip_model.get_image_features(**px)).float().cpu())
    return F.normalize(torch.cat(out), dim=-1)


@torch.no_grad()
def embed_texts(texts):
    toks = clip_tokenizer(texts, padding="max_length", max_length=CLIP_MAX_TOKENS,
                          truncation=True, return_tensors="pt").to(DEVICE)
    return F.normalize(pooled(clip_model.get_text_features(**toks)).float().cpu(), dim=-1)


TEXT_EMB = embed_texts(PROMPTS)
REAL_EMB = embed_images(REAL_IMAGES)
REFERENCE = float((TEXT_EMB * REAL_EMB).sum(-1).mean())


def batches(n, size):
    return [slice(i, min(i + size, n)) for i in range(0, n, size)]


def accepted_kwargs(pipe, kwargs):
    params = inspect.signature(pipe.__call__).parameters
    named = {k for k, p in params.items()
             if p.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)}
    return {k: v for k, v in kwargs.items() if k in named}


def run_arm(pipe, prompts, batch_size=BATCH_SIZE, **kwargs):
    accepted = accepted_kwargs(pipe, kwargs)
    images = []
    for part in batches(len(prompts), batch_size):
        images.extend(pipe(prompt=list(prompts)[part], **accepted).images)
    return images


def free(obj):
    try:
        obj.to("cpu")
    except Exception:
        pass
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def vram():
    if DEVICE.type != "cuda":
        return {}
    gb = 1024 ** 3
    return {"allocated_gb": round(torch.cuda.memory_allocated() / gb, 2),
            "peak_gb": round(torch.cuda.max_memory_allocated() / gb, 2)}


print(f"CLIP(text, real photo) = {REFERENCE:.4f}   <- the reference every arm is read against")

---
## 2. SD 1.5 — the reference arm

DPMSolver++ at 20 steps.

In [ ]:
panels, timings, notes = {}, {}, {}

pipe = None
for repo in SD_REPOS:
    try:
        pipe = StableDiffusionPipeline.from_pretrained(
            repo, torch_dtype=SD_DTYPE, safety_checker=None, requires_safety_checker=False)
        SD_REPO = repo
        break
    except Exception as exc:
        print(f"  {repo}: {type(exc).__name__}")
if pipe is None:
    raise RuntimeError("no Stable Diffusion repo could be loaded")

pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=True)
if DEVICE.type == "cuda":
    pipe.enable_attention_slicing()
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    dict(pipe.scheduler.config), algorithm_type="dpmsolver++", timestep_spacing="leading")

LABEL_SD = "SD 1.5 (CLIP, diffusion, 512)"
t0 = time.time()
panels[LABEL_SD] = run_arm(pipe, PROMPTS, num_inference_steps=SD_STEPS, guidance_scale=GUIDANCE,
                           generator=torch.Generator(device="cpu").manual_seed(SEED))
timings[LABEL_SD] = time.time() - t0
notes[LABEL_SD] = f"{SD_REPO}, DPMSolver++ {SD_STEPS} steps"

print(f"{timings[LABEL_SD]:.1f}s   {vram()}")
free(pipe)
del pipe

---
## 3. aMUSEd — the same encoder, a different mechanism

No `num_inference_steps` in the diffusion sense: aMUSEd unmasks tokens over a fixed number of
rounds. `run_arm` passes only what this pipeline declares, so nothing from SD's vocabulary leaks
across.

In [ ]:
amused = AmusedPipeline.from_pretrained(AMUSED_REPO, torch_dtype=SD_DTYPE).to(DEVICE)
amused.set_progress_bar_config(disable=True)

LABEL_AMUSED = "aMUSEd (CLIP, masked transformer, 256)"
t0 = time.time()
panels[LABEL_AMUSED] = run_arm(amused, PROMPTS, num_inference_steps=AMUSED_STEPS,
                               guidance_scale=GUIDANCE,
                               generator=torch.Generator(device="cpu").manual_seed(SEED))
timings[LABEL_AMUSED] = time.time() - t0
notes[LABEL_AMUSED] = f"{AMUSED_REPO}, {AMUSED_STEPS} unmasking rounds, CLIP ViT-L/14"

print(f"{timings[LABEL_AMUSED]:.1f}s   {vram()}")
free(amused)
del amused

---
## 4. LDM-BERT — the BERT-conditioned arm

Latent diffusion like SD, but conditioned on BERT. This is the BERT coverage without training
anything: a pretrained model that already does what a BERT-to-CLIP adapter would have to learn.

In [ ]:
ldm = LDMTextToImagePipeline.from_pretrained(LDM_REPO, torch_dtype=SD_DTYPE).to(DEVICE)
ldm.set_progress_bar_config(disable=True)

LABEL_LDM = "LDM-BERT (BERT, diffusion, 256)"
t0 = time.time()
panels[LABEL_LDM] = run_arm(ldm, PROMPTS, num_inference_steps=LDM_STEPS, guidance_scale=GUIDANCE,
                            generator=torch.Generator(device="cpu").manual_seed(SEED))
timings[LABEL_LDM] = time.time() - t0
notes[LABEL_LDM] = f"{LDM_REPO}, {LDM_STEPS} steps, BERT"

print(f"{timings[LABEL_LDM]:.1f}s   {vram()}")
free(ldm)
del ldm

---
## 5. Scores

In [ ]:
rows = []
for label, images in panels.items():
    scores = (TEXT_EMB * embed_images(images)).sum(-1).numpy()
    rows.append({"arm": label,
                 "clip_text_generated": round(float(scores.mean()), 4),
                 "std": round(float(scores.std()), 4),
                 "seconds": round(timings[label], 1),
                 "s_per_image": round(timings[label] / len(PROMPTS), 2),
                 "size": images[0].size[0]})

results = pd.DataFrame(rows).sort_values("clip_text_generated", ascending=False)
print(f"real COCO photographs score {REFERENCE:.4f}\n")
display(results)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.0))
y = np.arange(len(results))
ax.barh(y, results["clip_text_generated"], xerr=results["std"], capsize=4)
ax.axvline(REFERENCE, ls="--", c="grey", label=f"real photos ({REFERENCE:.3f})")
ax.set_yticks(y); ax.set_yticklabels(results["arm"], fontsize=8); ax.invert_yaxis()
ax.set_xlabel("CLIP(text, generated)")
ax.set_title("Prompt adherence across generative mechanisms")
ax.legend(); ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "m2_arm_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. The pictures

One row per arm, the same prompts down the columns. Tiles are resized to a common width so the rows
line up; the 256px arms genuinely produce smaller images, which the table records.

In [ ]:
UNSAFE = re.compile(r"[^A-Za-z0-9._-]+")
WIDTH, BAND = 256, 20


def slugify(text, max_length=60):
    return UNSAFE.sub("_", str(text)).strip("_")[:max_length] or "untitled"


def image_grid(images, rows=None, cols=None, pad=4):
    if rows is None and cols is None:
        cols = math.ceil(math.sqrt(len(images)))
    if cols is None:
        cols = math.ceil(len(images) / rows)
    if rows is None:
        rows = math.ceil(len(images) / cols)
    w, h = images[0].size
    canvas = Image.new("RGB", (cols * w + (cols + 1) * pad, rows * h + (rows + 1) * pad),
                       (255, 255, 255))
    for i, img in enumerate(images):
        r, c = divmod(i, cols)
        canvas.paste(img, (pad + c * (w + pad), pad + r * (h + pad)))
    return canvas


strips = {label: image_grid([img.resize((WIDTH, WIDTH)) for img in images], rows=1, cols=len(images))
          for label, images in panels.items()}
width = max(s.width for s in strips.values())
canvas = Image.new("RGB", (width, sum(s.height + BAND for s in strips.values())), (255, 255, 255))
draw, font, y = ImageDraw.Draw(canvas), ImageFont.load_default(), 0
for label, strip in strips.items():
    draw.text((6, y + 4), label, fill=(0, 0, 0), font=font)
    canvas.paste(strip, (0, y + BAND))
    y += BAND + strip.height

canvas.save(SWEEP_DIR / "arm_comparison.png")
print(SWEEP_DIR / "arm_comparison.png")

for label, images in panels.items():
    directory = SWEEP_DIR / slugify(label)
    directory.mkdir(parents=True, exist_ok=True)
    for i, (img, caption) in enumerate(zip(images, PROMPTS)):
        img.save(directory / f"{i:02d}_{slugify(caption)}.png")
    print(" ", directory)

canvas

In [ ]:
summary = {
    "prompts": PROMPTS,
    "seed": SEED,
    "protocol_note": ("shared prompts and seed, not shared latents -- aMUSEd has no starting "
                      "noise, so the fixed-latent protocol does not transfer"),
    "scoring_note": "CLIP ViT-B/32 judges; SD and aMUSEd are conditioned on ViT-L/14",
    "reference_real_photos": round(REFERENCE, 4),
    "arms": {row["arm"]: {**{k: v for k, v in row.items() if k != "arm"},
                          "detail": notes[row["arm"]]} for row in rows},
}
(TABLES_DIR / "milestone_2_comparison.json").write_text(json.dumps(summary, indent=2),
                                                        encoding="utf-8")

print(TABLES_DIR / "milestone_2_comparison.json")
print(f"\nbest adherence : {results.iloc[0]['arm']}  ({results.iloc[0]['clip_text_generated']:.4f})")
fastest = results.sort_values("s_per_image").iloc[0]
print(f"fastest        : {fastest['arm']}  ({fastest['s_per_image']:.2f}s per image)")

### What this shows

The sweep notebook spent its time on samplers and noise schedules. aMUSEd has none of those, and
still turns captions into pictures from the same CLIP embedding — so the noise schedule is a
property of *one way* of generating images, not of text-to-image generation as such.

LDM-BERT swaps the encoder instead of the mechanism, which is the other half of the BERT coverage
and needs no training to get.